# Laboratorio 07 — HBase: cuando los datos ya no caben en memoria

**Procesamiento Masivo de Datos (ELE051-B / EIN102B)** · USM 2026-2 · Paralelo 701
**Clase 9** · Viernes 9 de octubre de 2026 · Lab. de Informática 2

---

Ejercicios para trabajar en clase. No se entrega ni se evalúa: el notebook queda para ustedes.

El stream del Lab 06 guarda unas **7 horas** en ~4 MB de RAM; cinco años serían ~28 GB, y eso con 16 sensores. Hoy la historia va a **disco**, en HBase: el mismo generador del Lab 06, ahora con una semana completa (~1,4 millones de lecturas). Cinco pasos: el modelo de datos (Parte 1), la row key como único índice (Parte 2), el *hotspotting* (Parte 3), el camino de escritura visto en el disco (Parte 4) y, si el VPS conecta, el puente stream → HBase (Parte 5). La teoría está en el libreto de la Clase 8 (cátedra de HBase).

## Cómo trabajar este notebook

- En equipos de 2–3 personas, un notebook por equipo.
- **Todo es Python**, con la librería `happybase`, que habla con HBase por Thrift. Cada ejercicio indica el comando equivalente del `hbase shell`; si quieren verlo en consola: `docker compose exec hbase hbase shell`.
- Donde vean `___` hay que completar; cada celda trae una **pista**. La API completa: [documentación de happybase](https://happybase.readthedocs.io/en/latest/api.html).
- **HBase corre en su máquina**, en un contenedor (`docker-compose.yml` de esta carpeta), en modo *standalone*: HMaster, RegionServer y ZooKeeper en un solo proceso, guardando en un volumen en vez de HDFS. Las ideas son las de un cluster; la escala, la de un PC.
- La imagen `pmd-hbase` se construye **antes** de la clase (`docker compose build`, ~340 MB de descarga). Si no está, la Parte 0 la construye, pero tarda.
- La interfaz web del HMaster queda en <http://localhost:16010>: tablas, regiones y métricas. Vale la pena tenerla abierta.

## Configuración

Ejecútenla una vez. Usa el generador del Lab 06 (`../06/generador.py`, mismos 16 sensores, misma semilla) y, solo para la Parte 5, el `.env` del Lab 06 (clave del VPS y `EQUIPO`).

In [ ]:
import itertools
import json
import os
import subprocess
import sys
import time
import urllib.request
from datetime import datetime, timedelta, timezone

import happybase
import pandas as pd

sys.path.insert(0, "../06")                    # el generador del Lab 06
from generador import ESTACIONES, eventos

TZ_CL = timezone(timedelta(hours=-3))          # hora de Chile en octubre (UTC−3)
INICIO = datetime(2026, 10, 2, tzinfo=TZ_CL)   # la semana de histórico: 02-oct 00:00 → 08-oct 23:59
DIAS = 7
EV_POR_DIA = 172_800                           # 2 eventos/s × 86.400 s


def hbase_shell(comandos):
    """Ejecuta comandos en el `hbase shell` del contenedor; devuelve lo que importa de la salida."""
    p = subprocess.run(["docker", "compose", "exec", "-T", "hbase", "hbase", "shell", "-n"],
                       input=comandos, capture_output=True, text=True)
    ruido = ("SLF4J", "hbase:", "Took ", "HBase Shell", "Use \"help\"", "For Reference", "Version ")
    return "\n".join(l for l in (p.stdout + p.stderr).splitlines() if l.strip() and not l.startswith(ruido))


def en_contenedor(cmd):
    """Un comando de shell dentro del contenedor (para mirar el disco de HBase)."""
    return subprocess.run(["docker", "compose", "exec", "-T", "hbase", "sh", "-c", cmd],
                          capture_output=True, text=True).stdout


# --- Verificación del entorno ---
print("✅ happybase", happybase.__version__)
d = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
print(("✅ " + d.stdout.strip()) if d.returncode == 0 else "⚠️  `docker compose` no responde — sin Docker no hay laboratorio")
print("✅ generador del Lab 06:", len(ESTACIONES), "estaciones")
print(("✅ " if os.path.exists("../06/.env") else "⚠️  falta ") + "../06/.env (solo se usa en la Parte 5)")

---
# Parte 0 — Levantar HBase

`up -d --wait` levanta el contenedor y espera a que HBase tenga en línea su tabla de sistema (`hbase:meta`, el "índice de dónde vive cada región"). La primera vez tarda 30–60 s; si la imagen no estaba construida, varios minutos más.

In [ ]:
subprocess.run(["docker", "compose", "up", "-d", "--wait"], check=True)
conn = happybase.Connection("127.0.0.1", 9090)    # Thrift, el puerto 9090 del contenedor
print("✓ HBase responde · tablas:", conn.tables())

**Esperado:** `✓ HBase responde · tablas: []` la primera vez (si ya habían trabajado, aparecen sus tablas: el volumen las guarda). `conn` es **la conexión**: todo lo que sigue es `conn.<algo>` o `tabla.<algo>`.

---
# Parte 1 — El modelo de datos: un mapa ordenado de bytes

Cada celda de HBase es **(row key, familia:columna, timestamp) → bytes**. Las *familias* se declaran al crear la tabla; las columnas dentro de ellas, no.

### Ejercicio 1.1 — Crear una tabla

Creen `demo` con dos familias: `m` (medición, guardando hasta **3 versiones** por celda) y `meta` (descriptiva, con las opciones por defecto).

En consola: `create 'demo', {NAME => 'm', VERSIONS => 3}, 'meta'`

In [ ]:
if b"demo" in conn.tables():
    conn.delete_table("demo", disable=True)       # para poder re-ejecutar la celda
___   # Pista: conn.create_table("demo", {familia: dict(opciones), ...}) — max_versions=3 para "m"
demo = conn.table("demo")
print(conn.tables())

### Ejercicio 1.2 — `put` y `get`

Guarden una lectura de CC-001: `m:valor` = 42.5, `m:tipo` = flujo_vehicular y `meta:comuna` = Concepción, con la row key `CC-001#2026-10-09T08:15:00.000-03:00`. Léanla de vuelta.

En consola: `put 'demo', '<row key>', 'm:valor', '42.5'` · `get 'demo', '<row key>'`

In [ ]:
fila = b"CC-001#2026-10-09T08:15:00.000-03:00"
___   # Pista: demo.put(fila, {b"familia:columna": b"valor", ...}) — todo en bytes: "Concepción".encode()
___   # Pista: demo.row(fila)

**Esperado:** un dict `{b'm:valor': b'42.5', b'm:tipo': ..., b'meta:comuna': b'Concepci\xc3\xb3n'}`. **Todo son bytes**: la row key, los nombres de columna y los valores. HBase no sabe que `42.5` es un número ni que `Concepción` es texto en UTF-8 — eso lo decide quien lee. (Va a importar en el 2.4.)

### Ejercicio 1.3 — Versiones

Escriban dos valores más en la **misma celda** (`m:valor` = 43.0 y luego 44.1) y pidan las versiones guardadas, con su timestamp.

En consola: `get 'demo', '<row key>', {COLUMN => 'm:valor', VERSIONS => 3}`

In [ ]:
demo.put(fila, {b"m:valor": b"43.0"})
demo.put(fila, {b"m:valor": b"44.1"})
print("row() →", demo.row(fila)[b"m:valor"])
___   # Pista: demo.cells(fila, columna, versions=..., include_timestamp=True)

**✍️ Respuesta del equipo — 1.3:** ¿qué devuelve `row()` y qué `cells()`? ¿Qué pasaría con una cuarta escritura en la misma celda? `___`

### Ejercicio 1.4 — Disperso: cada fila con sus columnas

Agreguen una lectura de TAL-003 (calidad del aire) con `m:valor` = 61.2 y una columna que la de CC-001 no tiene: `m:pm10` = 88. Recorran la tabla.

En consola: `put 'demo', 'TAL-003#2026-10-09T08:15:00.500-03:00', 'm:pm10', '88'` · `scan 'demo'`

In [ ]:
___   # Pista: demo.put(row_key, {...}) con las dos columnas
for k, v in ___:   # Pista: demo.scan() recorre la tabla en orden de row key
    print(k, v)

**✍️ Respuesta del equipo — 1.4:** ¿cuántas columnas "tiene" la tabla `demo`? ¿En qué orden salieron las filas? `___`

---
# Parte 2 — Una semana de lecturas: la row key es el índice

### Ejercicio 2.1 — Tablas pre-divididas en regiones

Una **región** es un rango contiguo de row keys, servida por un RegionServer. HBase divide una tabla solo cuando crece (~10 GB), así que aquí la **pre-dividimos** en 4 regiones, como si tuviéramos 4 servidores. Dos tablas con los mismos datos y distinta row key:

| Tabla | Row key | Cortes entre regiones |
|---|---|---|
| `lecturas` | `estacion#timestamp` | `COR` · `PEN` · `TAL` (por estación) |
| `lecturas_ts` | `timestamp#estacion` | `2026-10-04` · `2026-10-06` · `2026-10-08` (por fecha) |

La creación con `SPLITS` no existe en happybase: va por el `hbase shell` del contenedor (dado).

In [ ]:
print(hbase_shell("""
create 'lecturas',    'm', SPLITS => ['COR', 'PEN', 'TAL']
create 'lecturas_ts', 'm', SPLITS => ['2026-10-04', '2026-10-06', '2026-10-08']
"""))
pd.DataFrame(conn.table("lecturas").regions())[["start_key", "end_key", "server_name"]]

**Esperado:** `Created table …` para ambas (o `ERROR: Table already exists` si re-ejecutan: no pasa nada) y 4 regiones de `lecturas`: `'' → COR`, `COR → PEN`, `PEN → TAL`, `TAL → ''`. Todas en el mismo servidor (`pmd-hbase`): es un PC.

### Ejercicio 2.2 — Cargar la semana

El generador del Lab 06 produce la semana completa (más una hora extra, la "en vivo" de la Parte 3). Completen `clave()`: la row key de `lecturas` es `estacion_id#timestamp`, en bytes.

In [ ]:
todo = list(itertools.islice(eventos(INICIO), DIAS * EV_POR_DIA + 7200))
semana, vivo = todo[:-7200], todo[-7200:]
print(f"{len(semana):,} eventos · {semana[0]['timestamp']} → {semana[-1]['timestamp']}")
print(f"{len(vivo):,} 'en vivo' · {vivo[0]['timestamp']} → {vivo[-1]['timestamp']}")


def clave(ev):
    return ___   # Pista: f"{estacion}#{timestamp}".encode() — HBase guarda bytes


def columnas(ev):
    return {b"m:valor": str(ev["valor"]).encode(), b"m:tipo": ev["tipo_sensor"].encode()}


def escribir(tabla, evs, fclave):
    """put de cada evento, en lotes de 2.000 (un viaje a HBase por lote, no por fila)."""
    with conn.table(tabla).batch(batch_size=2000) as b:
        for ev in evs:
            b.put(fclave(ev), columnas(ev))


print(clave(semana[0]))
t = time.time()
escribir("lecturas", semana, clave)
print(f"{len(semana):,} put en {time.time() - t:.1f} s")

**Esperado:** ~1,2 millones de eventos, del 02-oct 00:00 al 08-oct 23:59, cargados en unos segundos.

Ahora cuenten las filas. Contar en HBase es **recorrer** la tabla (no hay un `COUNT(*)` guardado); los filtros hacen que viaje solo la primera clave de cada fila. Tarda unos segundos:

In [ ]:
def contar(tabla):
    return sum(1 for _ in conn.table(tabla).scan(filter="FirstKeyOnlyFilter() AND KeyOnlyFilter()"))


n = contar("lecturas")
print(f"{n:,} filas · {len(semana) - n:,} put no crearon fila nueva")

**✍️ Respuesta del equipo — 2.2:** ¿por qué hay menos filas que `put`? (Pista: Lab 06, Ejercicio 3.1.) `___`

### Ejercicio 2.3 — `scan` por rango: la pregunta que la row key hace rápida

"Las lecturas de CC-001 entre las 09:00 y las 10:00 del 08-oct." Como las filas están **ordenadas por row key** y la key empieza por la estación, es un tramo **contiguo**: se pide desde dónde (`row_start`, incluido) hasta dónde (`row_stop`, excluido).

En consola: `scan 'lecturas', {STARTROW => 'CC-001#2026-10-08T09:00', STOPROW => 'CC-001#2026-10-08T10:00'}`

In [ ]:
t = time.time()
hora_cc = list(___)   # Pista: conn.table(...).scan(row_start=b"...", row_stop=b"...")
t_rango = time.time() - t
print(f"{len(hora_cc)} lecturas en {t_rango:.3f} s · promedio {sum(float(v[b'm:valor']) for _, v in hora_cc) / len(hora_cc):.1f} veh/min")
hora_cc[:3]

**Esperado:** unas 440 lecturas, en centésimas de segundo.

### Ejercicio 2.4 — La pregunta que la row key **no** hace rápida

"Todas las lecturas con `valor > 100`." El valor no está en la row key: HBase tiene que **recorrer la tabla entera** y filtrar celda por celda (`SingleColumnValueFilter`). La celda está dada; ejecútenla (tarda) y miren **dos** cosas: el tiempo, y cuántas filas devolvió.

In [ ]:
t = time.time()
devueltas = mayores_reales = 0
for _, v in conn.table("lecturas").scan(filter="SingleColumnValueFilter('m', 'valor', >, 'binary:100', true, true)"):
    devueltas += 1
    mayores_reales += float(v[b"m:valor"]) > 100
t_filtro = time.time() - t
print(f"filtro: {devueltas:,} filas en {t_filtro:.1f} s ({t_filtro / t_rango:,.0f}× el scan por rango)")
print(f"de esas, con valor > 100 de verdad: {mayores_reales:,}")

**✍️ Respuesta del equipo — 2.4:**
1. ¿Por qué el filtro es tanto más lento que el rango? ¿Qué pasaría con cinco años en vez de una semana? `___`
2. ¿Por qué el filtro devolvió casi todas las filas, si muy pocas pasan de 100? (Pista: Ejercicio 1.2.) `___`
3. Si "las lecturas sobre un umbral" fuera una pregunta frecuente, ¿qué harían? `___`

---
# Parte 3 — Hotspotting: la misma semana, otra row key

### Ejercicio 3.1 — Cargar `lecturas_ts`

Completen `clave_ts()`: `timestamp#estacion_id`. A primera vista parece la elección natural para datos que llegan en orden de tiempo.

In [ ]:
def clave_ts(ev):
    return ___   # Pista: como clave(), con el orden invertido


t = time.time()
escribir("lecturas_ts", semana, clave_ts)
print(f"{len(semana):,} put en {time.time() - t:.1f} s")

### Ejercicio 3.2 — Una hora en vivo: ¿qué región trabaja?

Ahora llega la hora "en vivo" (09-oct, 00:00–01:00) y se escribe en **las dos** tablas. ¿Cuántas escrituras recibió cada región? No lo calculamos nosotros: se lo preguntamos a HBase, que lleva la cuenta por región (sus métricas JMX, las mismas que muestra la interfaz web). HBase las publica cada ~10 s: la celda espera.

In [ ]:
URL_JMX = "http://localhost:16010/jmx?qry=Hadoop:service=HBase,name=RegionServer,sub=Regions"


def escrituras_por_region(tabla):
    """writeRequestCount de cada región de `tabla`, según las métricas del propio HBase."""
    m = json.load(urllib.request.urlopen(URL_JMX))["beans"][0]
    return {(r["start_key"].decode() or "(inicio)"):
            m[f"Namespace_default_table_{tabla}_region_{r['name'].decode().split('.')[-2]}_metric_writeRequestCount"]
            for r in conn.table(tabla).regions()}


time.sleep(12)                                   # que las métricas ya incluyan las cargas anteriores
antes = {t: escrituras_por_region(t) for t in ("lecturas", "lecturas_ts")}
escribir("lecturas", vivo, clave)
escribir("lecturas_ts", vivo, clave_ts)
time.sleep(25)                                   # HBase publica sus métricas cada ~10 s: se esperan dos ciclos
despues = {t: escrituras_por_region(t) for t in antes}
for t in antes:
    print(t, {r: despues[t][r] - antes[t][r] for r in antes[t]})

**Esperado:** en `lecturas`, las escrituras repartidas entre las 4 regiones; en `lecturas_ts`, ¿?

**✍️ Respuesta del equipo — 3.2:**
1. Con 4 RegionServers (uno por región), ¿cuántos trabajan durante la hora en vivo con cada diseño? `___`
2. ¿Se arregla agregando más servidores? ¿Por qué? `___`

### Ejercicio 3.3 — Timestamp invertido: lo más reciente, primero

"La última lectura de cada estación" (el dashboard del Lab 06). En `lecturas`, lo último de una estación está **al final** de su tramo. Con la key `estacion#(10¹³ − milisegundos)`, lo más nuevo tiene la key **más chica** y sale primero: basta un `scan` con `limit=1`.

Completen `clave_inv()` (el cálculo de los milisegundos está dado) y la consulta.

In [ ]:
def ms(ev):
    return int(datetime.fromisoformat(ev["timestamp"]).timestamp() * 1000)


def clave_inv(ev):
    return ___   # Pista: f"{estacion}#{10**13 - ms(ev):013d}" — el :013d rellena con ceros para que el orden de texto sea el numérico


if b"lecturas_inv" not in conn.tables():
    conn.create_table("lecturas_inv", {"m": dict()})
t = time.time()
escribir("lecturas_inv", todo, clave_inv)
print(f"{len(todo):,} put en {time.time() - t:.1f} s")

t = time.time()
ultimas = {}
for est, _ in ESTACIONES:
    k, v = next(___)   # Pista: conn.table(...).scan(row_prefix=b"<estacion>#", limit=1)
    ultimas[est] = {"valor": v[b"m:valor"].decode(), "tipo": v[b"m:tipo"].decode(), "ms": 10**13 - int(k.split(b"#")[1])}
print(f"16 últimas lecturas en {time.time() - t:.3f} s")
df = pd.DataFrame(ultimas).T
df["hora"] = pd.to_datetime(df.pop("ms").astype("int64"), unit="ms", utc=True).dt.tz_convert(TZ_CL).dt.strftime("%d-%m %H:%M:%S")
df

**✍️ Respuesta del equipo — 3.3:** ¿por qué `10**13 − ms`, y por qué rellenar a 13 dígitos? `___`

---
# Parte 4 — El camino de escritura, visto en el disco

Lo de la cátedra: `put` → **WAL** (log) + **MemStore** (RAM) → *flush* → **HFile** (archivo ordenado e inmutable) → *compaction*. En *standalone* todo eso está en el volumen, bajo `/data/hbase`: se puede mirar.

### Ejercicio 4.1 — ¿Dónde están los datos ahora?

In [ ]:
HFILES = """for r in /data/hbase/data/default/lecturas/*/m; do
  echo "región $(basename $(dirname $r) | cut -c1-8): $(ls $r | wc -l) HFiles, $(du -sh $r | cut -f1)"; done
echo "WAL: $(du -sh /data/hbase/WALs | cut -f1)" """
print(en_contenedor(HFILES))

**Esperado:** algunas regiones sin ninguna HFile. ¿Dónde están entonces sus filas?

### Ejercicio 4.2 — `flush`: de la MemStore al disco

En consola: `flush 'lecturas'`

In [ ]:
print(hbase_shell(___))   # Pista: el comando de consola, como texto
print(en_contenedor(HFILES))

**Esperado:** ahora cada región tiene al menos 1 HFile. Escriban otra vez la hora en vivo (mismas keys: son versiones nuevas) y vuelvan a hacer `flush`: los archivos anteriores **no se modifican**, aparece uno más por región.

In [ ]:
escribir("lecturas", vivo, clave)
print(hbase_shell("flush 'lecturas'"))
print(en_contenedor(HFILES))

### Ejercicio 4.3 — `major_compact`: fusionar

Varias HFiles por región hacen más lenta la lectura (hay que mirar en todas). La compactación mayor las fusiona en una, y de paso descarta versiones de sobra y lápidas de `delete`. Corre en segundo plano: la celda espera.

En consola: `major_compact 'lecturas'`

In [ ]:
print(hbase_shell(___))
time.sleep(25)
print(en_contenedor(HFILES))

**✍️ Respuesta del equipo — 4.3:** ¿por qué HBase escribe archivos nuevos en vez de modificar los existentes? ¿Qué se gana y qué se paga? `___`

### Ejercicio 4.4 — Apagar y prender

En el Lab 06, apagar el Redis del camino de rescate borraba todo. Aquí: reiniciar el contenedor y volver a contar.

In [ ]:
n_antes = contar("lecturas")
subprocess.run(["docker", "compose", "restart"], check=True)
subprocess.run(["docker", "compose", "up", "-d", "--wait"], check=True)
conn = happybase.Connection("127.0.0.1", 9090)   # la conexión anterior murió con el contenedor
print(f"antes: {n_antes:,} filas · después del reinicio: {contar('lecturas'):,}")

**Esperado:** el mismo número. HBase vive en disco (el volumen `pmd-hbase`); la RAM es solo la MemStore, y lo que estaba ahí se recupera del WAL o se vació al apagarse ordenadamente.

---
# Parte 5 — Del stream a HBase (si el VPS conecta)

El puente de la cátedra: un consumidor lee el stream del Lab 06 con su **propio consumer group** (`ingesta-<equipo>`), guarda cada lote en `lecturas` y **recién entonces** hace `XACK`. Los timestamps del VPS vienen en UTC: se pasan a hora de Chile para que las keys calcen con las de la semana.

In [ ]:
import redis

if os.path.exists("../06/.env"):
    for linea in open("../06/.env", encoding="utf-8"):
        k, _, v = linea.strip().partition("=")
        if k and not k.startswith("#"):
            os.environ[k.strip()] = v.strip()
EQUIPO = os.environ.get("EQUIPO", "equipo-1")
VPS_PORT = int(os.environ.get("VPS_PORT", 443))
try:
    r = redis.Redis(host=os.environ.get("VPS_HOST", "redis-stream.5.75.154.51.sslip.io"), port=VPS_PORT,
                    password=os.environ.get("VPS_PASS") or None, ssl=VPS_PORT == 443,
                    socket_connect_timeout=3, decode_responses=True)
    r.ping()
    print("✓ VPS conectado ·", EQUIPO)
except Exception as e:
    r = None
    print("✗ Sin VPS (", e, ") — la Parte 5 se ve en la proyección del profesor")

In [ ]:
STREAM, GRUPO = "sensores:flujo", f"ingesta-{EQUIPO}"


def a_hora_cl(ev):
    """El evento del VPS (timestamp en UTC) con el timestamp en hora de Chile, como los de la semana."""
    return {**ev, "timestamp": datetime.fromisoformat(ev["timestamp"]).astimezone(TZ_CL).isoformat(timespec="milliseconds")}


if r:
    try:
        r.xgroup_create(STREAM, GRUPO, id="$")
    except redis.ResponseError:
        pass                                          # ya existía: se re-ejecutó la celda
    tabla, ingestadas, fin = conn.table("lecturas"), 0, time.time() + 30
    while time.time() < fin:
        for _, entradas in r.xreadgroup(GRUPO, EQUIPO, {STREAM: ">"}, count=50, block=2000) or []:
            with tabla.batch() as b:
                for _, ev in entradas:
                    ev = a_hora_cl(ev)
                    ___   # Pista: lo mismo que escribir(): b.put(clave(ev), columnas(ev))
            ingestadas += ___   # Pista: r.xack(STREAM, GRUPO, *ids) — DESPUÉS del put
    print(f"{ingestadas} eventos: leídos del stream → guardados en HBase → confirmados · pendientes: {r.xpending(STREAM, GRUPO)['pending']}")
    print("el último, ya en HBase:", clave(ev), tabla.row(clave(ev)))

**✍️ Respuesta del equipo — 5:** ¿por qué el `XACK` va **después** del `put`? ¿Qué pasa si un evento se guarda dos veces? `___`

---
# Parte 6 — Síntesis del equipo

Tres líneas:

1. Los dos tiempos del 2.3 y el 2.4, y por qué uno no crece con la tabla y el otro sí. `___`
2. Si su **proyecto semestral** guardara su historia en HBase: ¿qué row key usarían, y qué pregunta haría rápida? `___`
3. Un dato de su proyecto que iría en **Redis** y uno que iría en **HBase**, y por qué. `___`

---
### Limpieza (antes de irse)

`docker compose down` apaga y borra el contenedor; el volumen `pmd-hbase` **queda** (sus tablas siguen ahí la próxima vez). En un PC compartido del laboratorio, bórrenlo también: son ~1,5 GB.

In [ ]:
subprocess.run(["docker", "compose", "down", "-v"], check=True)   # sin -v, los datos quedan en el volumen
print("Listo.")

---
*Fin del Laboratorio 07.*